# Phase 1 walkthrough

This notebook looks at the prepared data and the Phase 1 results. Run `python3 run_phase1.py` first so the `reports/` folder is filled.

In [1]:
import os
import sys

# work from the project folder so the imports and paths below work
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, ".")

import yaml
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, Markdown, display

from data_loader import prepare_dataset

with open("config.yaml") as f:
    config = yaml.safe_load(f)

df = prepare_dataset(config)

Loading prepared data from data/processed/m5_sample.csv


## 1. The data

In [2]:
print("Series:", df["series_id"].nunique())
print("Dates:", df["date"].min().date(), "to", df["date"].max().date())
print("Stores:", df["store_id"].nunique())
print()
print("Series per category:")
print(df.groupby("cat_id")["series_id"].nunique())

Series: 99
Dates: 2014-10-01 to 2016-05-22
Stores: 10

Series per category:
cat_id
FOODS        40
HOBBIES      25
HOUSEHOLD    34
Name: series_id, dtype: int64


In [3]:
active = df[df["is_active"] == 1]
summary = active.groupby("series_id").agg(
    mean_sales=("demand", "mean"),
    zero_share=("demand", lambda s: (s == 0).mean()),
    stockout_share=("is_stockout", "mean"),
)
print("Share of active days flagged as stockout:", round(active["is_stockout"].mean(), 3))
summary.describe().round(2)

Share of active days flagged as stockout: 0.195


,mean_sales,zero_share,stockout_share
count,99.00,99.00,99.00
mean,1.13,0.59,0.18
std,2.04,0.24,0.16
min,0.04,0.02,0.00
25%,0.29,0.38,0.07
50%,0.59,0.63,0.15
75%,1.28,0.79,0.23
max,17.74,0.97,0.81


## 2. One series with flagged stockouts

Grey dots are recorded sales. Red crosses are days flagged as stockouts: the sales number (0) is kept, but the day is marked so models do not treat it as real demand.

In [4]:
series_id = summary[(summary["mean_sales"] > 1) & (summary["stockout_share"].between(0.05, 0.3))].index[0]
one = df[df["series_id"] == series_id].tail(150)

plt.figure(figsize=(12, 3.5))
plt.plot(one["date"], one["demand"], "o", color="#52514e", markersize=3, label="sales")
flagged = one[one["is_stockout"] == 1]
plt.plot(flagged["date"], flagged["demand"], "x", color="#e34948", label="flagged stockout")
plt.title(series_id)
plt.legend()
plt.show()

## 3. Results

In [5]:
overall = pd.read_csv("reports/metrics_overall.csv")
overall[["model", "MAE", "RMSE", "WAPE", "mean_pinball", "coverage"]].round(3)

,model,MAE,RMSE,WAPE,mean_pinball,coverage
0,SeasonalNaive,1.302,2.568,0.921,0.420,0.836
1,KalmanFilter,1.029,1.865,0.728,0.321,0.878
2,GBDT,1.022,1.841,0.723,0.317,0.900


In [6]:
cold = pd.read_csv("reports/metrics_cold_start.csv")
cold[["segment", "model", "rows", "MAE", "WAPE", "mean_pinball", "coverage"]].round(3)

,segment,model,rows,MAE,WAPE,mean_pinball,coverage
0,cold start,SeasonalNaive,1341,1.156,1.037,0.374,0.818
1,established,SeasonalNaive,5952,1.335,0.901,0.430,0.840
2,cold start,KalmanFilter,1341,0.929,0.834,0.295,0.840
3,established,KalmanFilter,5952,1.052,0.710,0.327,0.886
4,cold start,GBDT,1341,0.915,0.821,0.297,0.883
5,established,GBDT,5952,1.046,0.706,0.322,0.904


In [7]:
display(Image("reports/error_by_horizon.png"))
display(Image("reports/forecast_example.png"))
display(Image("reports/kalman_tracking.png"))

## 4. Adding a model in Phase 2

`reports/predictions.csv` has one row per model, origin, series and day. A new model (DeepAR) only needs to produce the same columns (`series_id`, `date`, `step`, `point`, `q10`, `q50`, `q90`) for the series and origins listed in `reports/backtest_setup.json`; it is then scored exactly like the Phase 1 models.

In [8]:
predictions = pd.read_csv("reports/predictions.csv")
predictions.head()

,model,fold,origin,series_id,date,step,actual,point,q10,q50,q90,is_stockout,is_active,is_cold_start,days_since_launch_at_origin,is_scored
0,SeasonalNaive,1,2016-02-28,FOODS_2_007_TX_2,2016-02-29,1,0.0,1.0,1.0,1.0,1.0,0,1,0,804,1
1,SeasonalNaive,1,2016-02-28,FOODS_2_007_TX_2,2016-03-01,2,0.0,0.0,0.0,0.0,0.0,0,1,0,804,1
2,SeasonalNaive,1,2016-02-28,FOODS_2_007_TX_2,2016-03-02,3,0.0,0.0,0.0,0.0,0.0,0,1,0,804,1
3,SeasonalNaive,1,2016-02-28,FOODS_2_007_TX_2,2016-03-03,4,0.0,0.0,0.0,0.0,0.0,0,1,0,804,1
4,SeasonalNaive,1,2016-02-28,FOODS_2_007_TX_2,2016-03-04,5,1.0,0.0,0.0,0.0,0.0,0,1,0,804,1
